In [9]:
import os
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
import firebase_admin
from firebase_admin import credentials, db
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. CẤU HÌNH KẾT NỐI FIREBASE
# ==============================================================================
# Đường dẫn tới file JSON bản quyền tải từ Firebase về
FIREBASE_KEY_PATH = "serviceAccountKey.json" 
# Thay bằng link Realtime Database của bạn
FIREBASE_DB_URL = "https://edutrack-6db11-default-rtdb.firebaseio.com/" 

if not firebase_admin._apps:
    cred = credentials.Certificate(FIREBASE_KEY_PATH)
    firebase_admin.initialize_app(cred, {'databaseURL': FIREBASE_DB_URL})

print("[INFO] Đã kết nối thành công tới Firebase Cloud Database.")

# ==============================================================================
# 2. HÀM TẠO DỮ LIỆU GIẢ LẬP ĐỂ DEMO (NẾU FIREBASE CHƯA CÓ ĐỦ LOG TRONG 30 NGÀY)
# ==============================================================================
def generate_mock_data_to_firebase():
    """Tạo chuỗi thói quen đi học trong 20 buổi của 1 sinh viên để train LSTM, 
       và tạo ra 2 buổi cuối bị bất thường (gian lận) để xem AI bắt lỗi."""
    ref = db.reference('attendance_logs')
    ref.delete() # Xóa log cũ để tạo mới sạch sẽ
    
    uid_test = "1A2B3C4D"
    print(f"[INFO] Đang tạo dữ liệu giả lập chuỗi thời gian cho thẻ {uid_test}...")
    
    # Giả lập 20 ngày đi học bình thường (Check-in khoảng 07:00 - 07:15, học xong ra về sau 180 phút)
    for day in range(1, 21):
        # Tạo giờ check-in ngẫu nhiên quanh 7h sáng
        checkin_minute = np.random.randint(0, 15)
        timestamp_in = f"07:{checkin_minute:02d}:00"
        ref.push({
            "uid": uid_test,
            "status": "CHECK_IN",
            "timestamp": f"Day {day} - {timestamp_in}"
        })
        
        # Check-out sau đó khoảng 180 phút (3 tiếng)
        timestamp_out = f"10:{checkin_minute:02d}:00"
        ref.push({
            "uid": uid_test,
            "status": "CHECK_OUT",
            "timestamp": f"Day {day} - {timestamp_out}"
        })

    # Giả lập NGÀY BẤT THƯỜNG (Gian lận: Quẹt check-in rồi quẹt check-out ngay sau đó 2 phút để trốn về)
    ref.push({"uid": uid_test, "status": "CHECK_IN", "timestamp": "Day 21 - 07:05:00"})
    ref.push({"uid": uid_test, "status": "CHECK_OUT", "timestamp": "Day 21 - 07:07:00"}) # Bất thường!
    
    # Giả lập NGÀY BẤT THƯỜNG 2 (Đi học quá muộn, tận cuối ca mới đến quẹt thẻ)
    ref.push({"uid": uid_test, "status": "CHECK_IN", "timestamp": "Day 22 - 09:55:00"}) # Bất thường!
    ref.push({"uid": uid_test, "status": "CHECK_OUT", "timestamp": "Day 22 - 10:00:00"})
    
    print("[SUCCESS] Đã nạp chuỗi dữ liệu demo lên Firebase.")

# ==============================================================================
# 3. TRÍCH XUẤT ĐẶC TRƯNG CHUỖI THỜI GIAN (FEATURE ENGINEERING)
# ==============================================================================
def fetch_and_preprocess_data(target_uid):
    ref = db.reference('attendance_logs')
    logs = ref.get()
    
    if not logs:
        print("[ERROR] Không tìm thấy dữ liệu trên Firebase.")
        return None

    # Chuyển đổi dữ liệu JSON thành bảng Pandas DataFrame
    data_list = []
    for key, value in logs.items():
        if value['uid'] == target_uid:
            data_list.append(value)
            
    df = pd.DataFrame(data_list)
    
    # Tính toán khoảng cách thời gian giữa các cặp trạng thái IN - OUT của FSM
    processed_days = []
    
    # Nhóm dữ liệu theo ngày (Dựa vào chữ 'Day X' trong chuỗi timestamp mẫu)
    df['day_label'] = df['timestamp'].apply(lambda x: x.split(" - ")[0])
    df['time_str'] = df['timestamp'].apply(lambda x: x.split(" - ")[1])
    
    for day, group in df.groupby('day_label'):
        in_row = group[group['status'] == 'CHECK_IN']
        out_row = group[group['status'] == 'CHECK_OUT']
        
        if not in_row.empty and not out_row.empty:
            t_in = datetime.strptime(in_row['time_str'].values[0], "%H:%M:%S")
            t_out = datetime.strptime(out_row['time_str'].values[0], "%H:%M:%S")
            
            # Feature 1: Giờ Check-in quy đổi ra phút tính từ nửa đêm (ví dụ 7h05 = 425 phút)
            checkin_minutes = t_in.hour * 60 + t_in.minute
            # Feature 2: Tổng thời gian hiện diện thực tế trong lớp học (Thời gian học)
            duration = int((t_out - t_in).total_seconds() / 60)
            
            processed_days.append([checkin_minutes, duration])
            
    return np.array(processed_days)

# ==============================================================================
# 4. XÂY DỰNG VÀ HUẤN LUYỆN MẠNG LSTM AUTOENCODER
# ==============================================================================
def train_lstm_anomaly_detector(data):
    # Chuẩn hóa dữ liệu về khoảng [0, 1] để mạng mạng nơ-ron học tốt nhất
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data)
    
    # Biến đổi cấu trúc ma trận đầu vào cho phù hợp với định dạng chuỗi của LSTM [Samples, Timesteps, Features]
    # Ở đây mỗi sample là 1 ngày, timestep = 1, số lượng feature = 2
    X = scaled_data.reshape(scaled_data.shape[0], 1, scaled_data.shape[1])
    
    # Kiến trúc LSTM Autoencoder chuyên dùng để phát hiện bất thường ngầm
    model = Sequential([
        LSTM(16, activation='relu', input_shape=(X.shape[1], X.shape[2]), return_sequences=False),
        RepeatVector(X.shape[1]),
        LSTM(16, activation='relu', return_sequences=True),
        TimeDistributed(Dense(X.shape[2]))
    ])
    
    model.compile(optimizer='adam', loss='mae')
    
    # Chỉ huấn luyện mô hình dựa trên 20 ngày đầu tiên (là chuỗi hành vi đi học bình thường)
    print("[INFO] Đang tiến hành huấn luyện mạng nơ-ron hồi quy LSTM...")
    model.fit(X[:20], X[:20], epochs=100, batch_size=4, verbose=0)
    print("[SUCCESS] Mô hình LSTM đã tối ưu hóa xong trọng số.")
    
    # Dự đoán lại toàn bộ chuỗi để tính toán sai số tái tạo (Reconstruction Error)
    X_pred = model.predict(X)
    mae_loss = np.mean(np.abs(X_pred - X), axis=2).reshape(-1)
    
    # Thiết lập ngưỡng phát hiện lỗi (Threshold) dựa trên độ lệch lớn nhất của chuỗi bình thường
    threshold = np.max(mae_loss[:20]) * 1.5
    print(f"[INFO] Ngưỡng sai số thuật toán xác lập: {threshold:.4f}")
    
    return mae_loss, threshold

# ==============================================================================
# 5. CHẠY KIỂM CHỨNG VÀ ĐẨY CẢNH BÁO LÊN CLOUD FIREBASE
# ==============================================================================
if __name__ == "__main__":
    # Bước A: Đổ dữ liệu mẫu lên để kiểm thử
    generate_mock_data_to_firebase()
    
    # Bước B: Kéo dữ liệu về phân tích đặc trưng
    target_student = "1A2B3C4D"
    dataset = fetch_and_preprocess_data(target_student)
    
    if dataset is not None:
        # Bước C: Chạy mạng LSTM để chấm điểm sai số chuỗi thời gian
        losses, limit = train_lstm_anomaly_detector(dataset)
        
        # Bước D: Duyệt qua từng ngày để tìm điểm dị thường
        print("\n=== KẾT QUẢ PHÂN TÍCH CHUỖI HÀNH VI TỪ MÔ HÌNH AI ===")
        for idx, loss in enumerate(losses):
            day_num = idx + 1
            status_text = "BÌNH THƯỜNG"
            is_high_risk = False
            
            # Nếu sai số vượt ngưỡng tức là hành vi hôm đó lệch hoàn toàn khỏi thói quen bình thường
            if loss > limit:
                status_text = "CẢNH BÁO BẤT THƯỜNG (HIGH RISK)"
                is_high_risk = True
                
            print(f"Ngày {day_num:02d}: Giờ check-in={dataset[idx][0]}p, Thời lượng học={dataset[idx][1]}p -> Sai số AI: {loss:.4f} [{status_text}]")
            
            # Đẩy cờ cảnh báo trực tiếp lên Firebase tại hồ sơ sinh viên đó
            student_profile_ref = db.reference(f'student_profiles/{target_student}/ai_analysis/day_{day_num}')
            student_profile_ref.set({
                "checkin_minutes_from_midnight": int(dataset[idx][0]),
                "class_duration_minutes": int(dataset[idx][1]),
                "ai_reconstruction_error": float(loss),
                "is_high_risk_anomaly": is_high_risk,
                "last_evaluated": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
        print("\n[SUCCESS] Toàn bộ nhãn cảnh báo từ mạng LSTM đã đồng bộ trực tiếp lên Firebase Cloud!")

FileNotFoundError: [Errno 2] No such file or directory: 'serviceAccountKey.json'